In [266]:
import pandas as pd
import time

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

from sklearn.metrics import accuracy_score, classification_report

In [267]:
# Read the dataset
df = pd.read_csv(r'D:\Downloads\ml_features_and_labels.csv')

# Use the provided split column to create train/test sets and exclude metadata
# columns that would leak information about the label.
feature_cols = [c for c in df.columns if c not in ['label', 'ID', 'split', 'taxonomy']]
df = df.fillna(0)

In [268]:
lis = df.select_dtypes(include=['bool'])
le = LabelEncoder()
for col in lis:
    df[col] = le.fit_transform(df[col])

In [269]:
cols_to_drop = ['taxonomy','ID','split'] 
df = df.drop(columns=cols_to_drop)

In [270]:

normal_data = df[df['label'] == 0]
anomaly_data = df[df['label'] == 1]

# 2. Split NORMAL data into training and testing sets
X_normal_train, X_normal_test = train_test_split(
    normal_data.drop('label', axis=1), 
    test_size=0.2, 
    random_state=42
)

X_test = pd.concat([X_normal_test, anomaly_data.drop('label', axis=1)])
y_test = pd.concat([
    pd.Series([0] * len(X_normal_test)), # True labels for normal is now 0
    pd.Series([1] * len(anomaly_data))   # True labels for anomalies is now 1
])

# 4. Initialize and train the One-Class SVM
model = OneClassSVM(kernel='rbf', gamma='auto', nu=0.01)
model.fit(X_normal_train)

# 5. Make predictions on the mixed test set
raw_predictions = model.predict(X_test)
# Scikit-learn outputs -1 for anomaly and 1 for normal.
# Your dataset uses 1 for anomaly and 0 for normal.
final_predictions = [1 if p == -1 else 0 for p in raw_predictions]

# 7. Evaluate the model
print(classification_report(y_test, final_predictions))

              precision    recall  f1-score   support

           0       0.88      0.99      0.93      7000
           1       0.98      0.81      0.89      5010

    accuracy                           0.91     12010
   macro avg       0.93      0.90      0.91     12010
weighted avg       0.92      0.91      0.91     12010



In [271]:
# 'contamination' is the equivalent of 'nu' in One-Class SVM. 
# It sets the expected proportion of outliers.
model = IsolationForest(
    n_estimators=500,      # Number of trees in the forest
    contamination=0.05,    # Assuming 5% of training data might be noise
    random_state=42        # Ensures reproducible results
)

# Train on the UN-SCALED normal data
model.fit(X_normal_train)

raw_predictions = model.predict(X_test)

# Scikit-learn still outputs -1 for anomaly and 1 for inlier.
# We map it back so 1 = anomaly, 0 = normal.
final_predictions = [1 if p == -1 else 0 for p in raw_predictions]

# 7. Evaluate the model
print(classification_report(y_test, final_predictions))

              precision    recall  f1-score   support

           0       0.70      0.95      0.81      7000
           1       0.87      0.44      0.58      5010

    accuracy                           0.74     12010
   macro avg       0.78      0.69      0.69     12010
weighted avg       0.77      0.74      0.71     12010



In [272]:
scaler = StandardScaler()
X_normal_train_scaled = scaler.fit_transform(X_normal_train)
X_test_scaled = scaler.transform(X_test)

model = LocalOutlierFactor(
    n_neighbors=10,       # How many neighbors to look at to determine density
    contamination=0.03,   # Expected proportion of outliers in training data
    novelty=True          # CRITICAL: Must be True to use .predict() on new test data
)

# Train on the SCALED normal data
model.fit(X_normal_train_scaled)

raw_predictions = model.predict(X_test_scaled)


# Scikit-learn outputs -1 for anomaly and 1 for inlier.
# We map it back so 1 = anomaly, 0 = normal.
final_predictions = [1 if p == -1 else 0 for p in raw_predictions]

# 7. Evaluate the model
print(classification_report(y_test, final_predictions))


              precision    recall  f1-score   support

           0       0.88      0.97      0.92      7000
           1       0.95      0.81      0.87      5010

    accuracy                           0.90     12010
   macro avg       0.91      0.89      0.90     12010
weighted avg       0.91      0.90      0.90     12010

